# 00 Real Data Loading Smoke Test

Run this notebook top-to-bottom. It checks that the real Hugging Face datasets in `config/experiment.yaml` can be constructed, sampled, and batched.

No synthetic fixture is used here.

## 1. Imports and Project Path

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/Users/similoluwaokunowo/Desktop/Research/geoai-lightning-talk-research')

## 2. Harness Imports

In [3]:
from eval_harness.config import load_yaml, dataset_config_path
from eval_harness.datasets import build_dataset

## 3. Load Experiment Config

In [4]:
experiment = load_yaml(PROJECT_ROOT / 'config' / 'experiment.yaml')
dataset_names = experiment['datasets']
dataset_names

['sen1floods11_ghana', 'ftw_africa']

## 4. Define Notebook Helpers

In [5]:
def collate_samples(samples):
    xs = [torch.as_tensor(sample['x']) for sample in samples]
    ys = [torch.as_tensor(sample['y']) for sample in samples]
    return {
        'id': [sample['id'] for sample in samples],
        'x': torch.stack(xs),
        'y': torch.stack(ys) if all(y.shape == ys[0].shape for y in ys) else ys,
    }


def batch_y_shape(batch):
    return tuple(batch['y'].shape) if hasattr(batch['y'], 'shape') else 'list'


def summarize_sample(name, split, ds, sample, batch):
    return {
        'dataset': name,
        'split': split,
        'n': len(ds),
        'sample_id': sample['id'],
        'x_shape': tuple(sample['x'].shape),
        'x_dtype': str(sample['x'].dtype),
        'x_min': float(np.min(sample['x'])),
        'x_mean': float(np.mean(sample['x'])),
        'x_max': float(np.max(sample['x'])),
        'y_shape': tuple(np.shape(sample['y'])),
        'y_dtype': str(np.asarray(sample['y']).dtype),
        'batch_x_shape': tuple(batch['x'].shape),
        'batch_y_shape': batch_y_shape(batch),
    }

## 5. Dataset Overview From Config

In [6]:
dataset_configs = {
    name: load_yaml(dataset_config_path(name))
    for name in dataset_names
}

overview_rows = []
for name, cfg in dataset_configs.items():
    overview_rows.append({
        'dataset': name,
        'display_name': cfg['display_name'],
        'region': cfg['region'],
        'theme': cfg['climate_theme'],
        'task_type': cfg['task_type'],
        'input_shape': tuple(cfg['input']['shape']),
        'loader': cfg['data']['loader'],
        'hf_repo': cfg['data'].get('hf_repo'),
    })

pd.DataFrame(overview_rows)

,dataset,display_name,region,theme,task_type,input_shape,loader,hf_repo
0,sen1floods11_ghana,Sen1Floods11 Ghana,Ghana,Disaster response and humanitarian flood mapping,segmentation,"(13, 512, 512)",hf_sen1floods11,KozaMateusz/sen1floods11
1,ftw_africa,Fields of the Planet Africa,"Kenya, Rwanda, South Africa",Food security and agricultural field-boundary ...,segmentation,"(8, 512, 512)",hf_ftw_planet,taylor-geospatial/ftw-planet


## 6. Build Real Datasets

In [7]:
loaded = {}
for name, cfg in dataset_configs.items():
    print(f'Building {name}')
    loaded[name] = {
        'cfg': cfg,
        'splits': {
            split: build_dataset(cfg, split=split)
            for split in ['train', 'val', 'test']
        },
    }

list(loaded)

Building sen1floods11_ghana
Building ftw_africa


dataset/south_africa.tar:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

['sen1floods11_ghana', 'ftw_africa']

## 7. Sample and Batch Smoke Test

In [8]:
summary_rows = []

for name, bundle in loaded.items():
    print(f'\nDATASET: {name}')
    for split, ds in bundle['splits'].items():
        sample = ds[0]
        loader = DataLoader(
            ds,
            batch_size=min(2, len(ds)),
            shuffle=False,
            collate_fn=collate_samples,
        )
        batch = next(iter(loader))
        row = summarize_sample(name, split, ds, sample, batch)
        summary_rows.append(row)
        print(
            f"  {split}: n={row['n']} "
            f"batch_x={row['batch_x_shape']} "
            f"batch_y={row['batch_y_shape']}"
        )


DATASET: sen1floods11_ghana
  train: n=37 batch_x=(2, 13, 512, 512) batch_y=(2, 512, 512)


v1.1/data/flood_events/HandLabeled/S2Han(…):   0%|          | 0.00/2.03M [00:00<?, ?B/s]

v1.1/data/flood_events/HandLabeled/Label(…):   0%|          | 0.00/1.01k [00:00<?, ?B/s]

v1.1/data/flood_events/HandLabeled/S2Han(…):   0%|          | 0.00/2.22M [00:00<?, ?B/s]

v1.1/data/flood_events/HandLabeled/Label(…):   0%|          | 0.00/1.01k [00:00<?, ?B/s]

...

v1.1/data/flood_events/HandLabeled/S2Han(…):   0%|          | 0.00/2.29M [00:00<?, ?B/s]

v1.1/data/flood_events/HandLabeled/Label(…):   0%|          | 0.00/1.01k [00:00<?, ?B/s]

v1.1/data/flood_events/HandLabeled/S2Han(…):   0%|          | 0.00/2.54M [00:00<?, ?B/s]

v1.1/data/flood_events/HandLabeled/Label(…):   0%|          | 0.00/5.24k [00:00<?, ?B/s]

  test: n=8 batch_x=(2, 13, 512, 512) batch_y=(2, 512, 512)

DATASET: ftw_africa
  train: n=796 batch_x=(2, 8, 512, 512) batch_y=(2, 512, 512)
  val: n=170 batch_x=(2, 8, 512, 512) batch_y=(2, 512, 512)
  test: n=171 batch_x=(2, 8, 512, 512) batch_y=(2, 512, 512)


## 8. Results

In [9]:
summary = pd.DataFrame(summary_rows)
summary

,dataset,split,n,sample_id,x_shape,x_dtype,x_min,x_mean,x_max,y_shape,y_dtype,batch_x_shape,batch_y_shape
0,sen1floods11_ghana,train,37,Ghana_103272,"(13, 512, 512)",float32,44.0,1935.603027,5658.0000,"(512, 512)",int64,"(2, 13, 512, 512)","(2, 512, 512)"
1,sen1floods11_ghana,val,8,Ghana_49890,"(13, 512, 512)",float32,20.0,1536.012939,4260.0000,"(512, 512)",int64,"(2, 13, 512, 512)","(2, 512, 512)"
2,sen1floods11_ghana,test,8,Ghana_83483,"(13, 512, 512)",float32,33.0,2308.002930,4373.0000,"(512, 512)",int64,"(2, 13, 512, 512)","(2, 512, 512)"
3,ftw_africa,train,796,south_africa/g1_00008_1,"(8, 512, 512)",float32,0.0,0.098763,0.4287,"(512, 512)",int64,"(2, 8, 512, 512)","(2, 512, 512)"
4,ftw_africa,val,170,south_africa/g3_00006_15,"(8, 512, 512)",float32,0.0,0.074744,0.4748,"(512, 512)",int64,"(2, 8, 512, 512)","(2, 512, 512)"
5,ftw_africa,test,171,kenya/g0_0000008704-0000003584,"(8, 512, 512)",float32,0.0,0.133480,0.4817,"(512, 512)",int64,"(2, 8, 512, 512)","(2, 512, 512)"


## Pass Criteria

- `sen1floods11_ghana` loads real Sentinel-2 image/mask pairs from Hugging Face.
- `ftw_africa` loads downloaded FTW-Planet Africa country shards from Hugging Face.
- Train/val/test splits instantiate and produce batch tensors.
- FTW samples have 8-channel image tensors and 3-class segmentation masks.
